# Dados de Entrada
* Selecione "Adicionar ao Drive"

* Dados adicionados na aula anterior:
  * https://tinyurl.com/bigdata-amz





## Acesso ao Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Setup

## Instalação de pacotes

In [2]:
!pip install pyspark

## Preparação do ambiente

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import *

from datetime import datetime

appName = 'Big Data SQL'
master = 'local[*]'

spark = SparkSession.builder     \
    .master(master) \
    .appName(appName) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")



# Extract: Leitura de Dados Brutos / CSV

In [4]:
#amz = spark.read.csv("/content/drive/My Drive/amz/Toys_and_Games.csv", header=False, inferSchema=True)
amz = spark.read.csv("/content/drive/My Drive/amz/small.csv", header=False, inferSchema=True)


In [10]:
!head "/content/drive/My Drive/amz/small.csv"

0020232233,A1IDMI31WEANAF,2.0,1474502400
0020232233,A4BCEVVZ4Y3V3,1.0,1474156800
0020232233,A2EZ9PY1IHHBX0,3.0,1473638400
0020232233,A139PXTTC2LGHZ,5.0,1488412800
0020232233,A3IB33V29XIL8O,1.0,1486512000
0020232233,A1J86V48S4KRJE,5.0,1485475200
0020232233,A14J12PRBLGHF4,5.0,1483315200
0020232233,A2UKOWP9ICU416,5.0,1481932800
0020232233,A2ONKKDETRWT79,4.0,1481760000
0020232233,AK9GN9KZZNTEP,3.0,1481241600


In [8]:
amz

DataFrame[_c0: string, _c1: string, _c2: double, _c3: int]

In [9]:
amz.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: double (nullable = true)
 |-- _c3: integer (nullable = true)



In [11]:
amz.show(5)


+----------+--------------+---+----------+
|       _c0|           _c1|_c2|       _c3|
+----------+--------------+---+----------+
|0020232233|A1IDMI31WEANAF|2.0|1474502400|
|0020232233| A4BCEVVZ4Y3V3|1.0|1474156800|
|0020232233|A2EZ9PY1IHHBX0|3.0|1473638400|
|0020232233|A139PXTTC2LGHZ|5.0|1488412800|
|0020232233|A3IB33V29XIL8O|1.0|1486512000|
+----------+--------------+---+----------+
only showing top 5 rows


In [12]:
amz.describe().show()

+-------+-------------------+--------------------+------------------+--------------------+
|summary|                _c0|                 _c1|               _c2|                 _c3|
+-------+-------------------+--------------------+------------------+--------------------+
|  count|             500000|              500000|            500000|              500000|
|   mean|1.960764659029618E9|                NULL|          4.287652|   1.3795242776064E9|
| stddev| 2.00231205222927E9|                NULL|1.2097318933542653|1.1276813053443997E8|
|    min|         0020232233|A0011756FPL8K71Q5TAQ|               1.0|           939168000|
|    max|         B000HDH02Q|       AZZZZS162JNL0|               5.0|          1526256000|
+-------+-------------------+--------------------+------------------+--------------------+



# Transform: Operações com dados

## Renomeação de Colunas

In [13]:
amz = amz.toDF("item", "user", "rating", "timestamp")
amz.printSchema()

root
 |-- item: string (nullable = true)
 |-- user: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [14]:
amz.show()

+----------+--------------+------+----------+
|      item|          user|rating| timestamp|
+----------+--------------+------+----------+
|0020232233|A1IDMI31WEANAF|   2.0|1474502400|
|0020232233| A4BCEVVZ4Y3V3|   1.0|1474156800|
|0020232233|A2EZ9PY1IHHBX0|   3.0|1473638400|
|0020232233|A139PXTTC2LGHZ|   5.0|1488412800|
|0020232233|A3IB33V29XIL8O|   1.0|1486512000|
|0020232233|A1J86V48S4KRJE|   5.0|1485475200|
|0020232233|A14J12PRBLGHF4|   5.0|1483315200|
|0020232233|A2UKOWP9ICU416|   5.0|1481932800|
|0020232233|A2ONKKDETRWT79|   4.0|1481760000|
|0020232233| AK9GN9KZZNTEP|   3.0|1481241600|
|0020232233|A26HO01PDWN6O5|   5.0|1481068800|
|0020232233| A8PF4X87CS3ZZ|   5.0|1478304000|
|0020232233|A28QGCUNS22JPH|   5.0|1478217600|
|038536539X|A2X2RJWNAV4FM0|   2.0|1492992000|
|038536539X|A1DIRJ8YNI5TKL|   2.0|1492387200|
|038536539X|A2G5IDD919P2DK|   5.0|1485388800|
|0486277577| AGCAAWP1AVVZR|   4.0|1079395200|
|0486277577| AD2CIUI4TSVY7|   5.0|1019952000|
|0486277577| APTEPFKL03QA0|   5.0|

## Operações entre colunas

In [15]:
# Coluna nova: data criada a partir do timestamp

amz = amz.withColumn("date", to_date(from_unixtime("timestamp")))
amz.printSchema()
amz.show(10)




root
 |-- item: string (nullable = true)
 |-- user: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)
 |-- date: date (nullable = true)

+----------+--------------+------+----------+----------+
|      item|          user|rating| timestamp|      date|
+----------+--------------+------+----------+----------+
|0020232233|A1IDMI31WEANAF|   2.0|1474502400|2016-09-22|
|0020232233| A4BCEVVZ4Y3V3|   1.0|1474156800|2016-09-18|
|0020232233|A2EZ9PY1IHHBX0|   3.0|1473638400|2016-09-12|
|0020232233|A139PXTTC2LGHZ|   5.0|1488412800|2017-03-02|
|0020232233|A3IB33V29XIL8O|   1.0|1486512000|2017-02-08|
|0020232233|A1J86V48S4KRJE|   5.0|1485475200|2017-01-27|
|0020232233|A14J12PRBLGHF4|   5.0|1483315200|2017-01-02|
|0020232233|A2UKOWP9ICU416|   5.0|1481932800|2016-12-17|
|0020232233|A2ONKKDETRWT79|   4.0|1481760000|2016-12-15|
|0020232233| AK9GN9KZZNTEP|   3.0|1481241600|2016-12-09|
+----------+--------------+------+----------+----------+
only showi

In [16]:
amz = amz.drop("timestamp")


In [17]:
amz.printSchema()
amz.show()

root
 |-- item: string (nullable = true)
 |-- user: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- date: date (nullable = true)

+----------+--------------+------+----------+
|      item|          user|rating|      date|
+----------+--------------+------+----------+
|0020232233|A1IDMI31WEANAF|   2.0|2016-09-22|
|0020232233| A4BCEVVZ4Y3V3|   1.0|2016-09-18|
|0020232233|A2EZ9PY1IHHBX0|   3.0|2016-09-12|
|0020232233|A139PXTTC2LGHZ|   5.0|2017-03-02|
|0020232233|A3IB33V29XIL8O|   1.0|2017-02-08|
|0020232233|A1J86V48S4KRJE|   5.0|2017-01-27|
|0020232233|A14J12PRBLGHF4|   5.0|2017-01-02|
|0020232233|A2UKOWP9ICU416|   5.0|2016-12-17|
|0020232233|A2ONKKDETRWT79|   4.0|2016-12-15|
|0020232233| AK9GN9KZZNTEP|   3.0|2016-12-09|
|0020232233|A26HO01PDWN6O5|   5.0|2016-12-07|
|0020232233| A8PF4X87CS3ZZ|   5.0|2016-11-05|
|0020232233|A28QGCUNS22JPH|   5.0|2016-11-04|
|038536539X|A2X2RJWNAV4FM0|   2.0|2017-04-24|
|038536539X|A1DIRJ8YNI5TKL|   2.0|2017-04-17|
|038536539X|A2G5IDD919

In [21]:
amz.count()

500000

## Ordenação

In [22]:
# Observação: evitar ordenar dados brutos como no exemplo abaixo :-)
amz.sort(col('date')).show(10)

+----------+--------------+------+----------+
|      item|          user|rating|      date|
+----------+--------------+------+----------+
|1572810939|A1DTOHMM2Y5KY0|   5.0|1999-10-06|
|B00000GBQL| AXH34TXSUTQK1|   5.0|1999-12-11|
|157982319X| AKALUPQJFJVZ0|   5.0|1999-12-11|
|157982319X|A3370XCM3EVNLH|   5.0|1999-12-19|
|B00000IVQB|A36MQW22BD8LJU|   5.0|1999-12-27|
|157982319X|A3IMU6K5OOWYAJ|   5.0|1999-12-28|
|B00000JIK2| ACLSB702RPUMZ|   5.0|1999-12-28|
|157982319X|A1XTKU9E0443RF|   5.0|2000-02-09|
|B00000IWFB| AIL4HD8BRUHV9|   5.0|2000-03-08|
|B00000J419|A2S07LNFU0W0U3|   4.0|2000-04-16|
+----------+--------------+------+----------+
only showing top 10 rows


In [23]:
amz.sort(desc('date')).show(10)

+----------+--------------+------+----------+
|      item|          user|rating|      date|
+----------+--------------+------+----------+
|B00004TFT1| AW6KP97K31FLB|   1.0|2018-05-14|
|B00004YO0Y|A1316QDPGPLQGM|   3.0|2018-05-13|
|B00004TFT1|A3VICMWKE9FQ2S|   5.0|2018-05-13|
|B00004YO0Y| AEMEYXG4NYHPM|   5.0|2018-05-13|
|B00068Q7LC|A2O5C5TBIMB6YR|   5.0|2018-05-12|
|B00005O6B7|A3A5KR27F7VHLX|   5.0|2018-05-11|
|B00068Q7LC| AG0G5QKWVJQ1J|   4.0|2018-05-11|
|B00004YO0Y| A1QICE026I0QI|   5.0|2018-05-11|
|B000197NXM|A2GGS9PIR8T67G|   1.0|2018-05-11|
|B00004YO0Y| A81FSOR6SFRUX|   5.0|2018-05-11|
+----------+--------------+------+----------+
only showing top 10 rows


## Filtros Básicos

In [24]:
amz.count()

500000

In [25]:
before_2000 = amz.filter(amz.date < "2000-01-01")


In [26]:
before_2000.count()

7

In [27]:
before_2000.show()

+----------+--------------+------+----------+
|      item|          user|rating|      date|
+----------+--------------+------+----------+
|1572810939|A1DTOHMM2Y5KY0|   5.0|1999-10-06|
|157982319X|A3IMU6K5OOWYAJ|   5.0|1999-12-28|
|157982319X|A3370XCM3EVNLH|   5.0|1999-12-19|
|157982319X| AKALUPQJFJVZ0|   5.0|1999-12-11|
|B00000GBQL| AXH34TXSUTQK1|   5.0|1999-12-11|
|B00000IVQB|A36MQW22BD8LJU|   5.0|1999-12-27|
|B00000JIK2| ACLSB702RPUMZ|   5.0|1999-12-28|
+----------+--------------+------+----------+



In [29]:
# Mesmo resultado, apenas sintaxe diferente. Preciso diferenciar string de coluna
before_2000 = amz.filter(col("date") < "2000-01-01")


In [ ]:
# Não funciona, precisa filtrar a partir de condição com uma coluna
# before_2000 = amz.filter("date" < "2000-01-01")


In [30]:
before_2000.count()

7

In [31]:
before_2000.show()

+----------+--------------+------+----------+
|      item|          user|rating|      date|
+----------+--------------+------+----------+
|1572810939|A1DTOHMM2Y5KY0|   5.0|1999-10-06|
|157982319X|A3IMU6K5OOWYAJ|   5.0|1999-12-28|
|157982319X|A3370XCM3EVNLH|   5.0|1999-12-19|
|157982319X| AKALUPQJFJVZ0|   5.0|1999-12-11|
|B00000GBQL| AXH34TXSUTQK1|   5.0|1999-12-11|
|B00000IVQB|A36MQW22BD8LJU|   5.0|1999-12-27|
|B00000JIK2| ACLSB702RPUMZ|   5.0|1999-12-28|
+----------+--------------+------+----------+



In [32]:
in_the_year_2000 = amz.filter((amz.date >= "2000-01-01") & (amz.date < "2001-01-01"))


In [33]:
in_the_year_2000.count()

502

In [34]:
in_the_year_2000.show()

+----------+--------------+------+----------+
|      item|          user|rating|      date|
+----------+--------------+------+----------+
|0486277577|A14FXEKB1PXTZG|   5.0|2000-10-15|
|0963469150|A13YEV0INF38CS|   5.0|2000-11-07|
|0963469150|A3ASMCU6UXK6AT|   5.0|2000-09-12|
|0963469150|A25V2TQWJ5ZJ7E|   4.0|2000-08-26|
|1556343841|A3COBCG6X0692Y|   4.0|2000-12-25|
|157982319X|A3J9OINT21H7WN|   5.0|2000-05-25|
|157982319X|A1XTKU9E0443RF|   5.0|2000-02-09|
|157982319X|A2RH1TQ6SQBYJU|   5.0|2000-12-30|
|157982319X|A1OOB7V0HWGR9I|   5.0|2000-12-28|
|157982319X|A2H1AV74Z44F73|   5.0|2000-12-22|
|157982319X| AXBLVTFGIY66S|   5.0|2000-12-18|
|157982319X|A1KTR2YNGAXTNZ|   5.0|2000-12-10|
|157982319X|A3O5SDP2X761HW|   5.0|2000-12-08|
|157982319X|A3EJC048BLUXYP|   5.0|2000-12-08|
|157982319X|A2WFDA7XR4B4AP|   5.0|2000-11-23|
|157982319X|A1HHNT9OP7WQ49|   5.0|2000-11-13|
|157982319X|A2ZL0LYJ9YZ54E|   5.0|2000-10-27|
|157982319X|A3NFDFYQAKS0SW|   5.0|2000-08-31|
|157982319X|A2GFC0OQV2T8AD|   5.0|

In [38]:
amz.count()

500000

In [37]:
in_the_year_2000.sort(in_the_year_2000.date).show()

+----------+--------------+------+----------+
|      item|          user|rating|      date|
+----------+--------------+------+----------+
|157982319X|A1XTKU9E0443RF|   5.0|2000-02-09|
|B00000IWFB| AIL4HD8BRUHV9|   5.0|2000-03-08|
|B00000J419|A2S07LNFU0W0U3|   4.0|2000-04-16|
|B00000IWD2|A1UXMT6XQFZLA1|   5.0|2000-04-21|
|B00000IW4C|A35JE1NQ0HL8RQ|   3.0|2000-05-02|
|157982319X|A3J9OINT21H7WN|   5.0|2000-05-25|
|B00004SDAG|A2S07LNFU0W0U3|   4.0|2000-05-31|
|B00000IWGE|A1HO9J4DCQDGP9|   5.0|2000-06-09|
|B00002S8AQ|A1HO9J4DCQDGP9|   5.0|2000-06-09|
|157982319X| AT2BYFEVLVITT|   5.0|2000-06-10|
|157982319X|A3869M2QZA4NE5|   5.0|2000-06-16|
|B00002S8AQ|A1IPKJKUUMSL2A|   5.0|2000-07-06|
|B00000IWFC|A1ETV7PW3286IM|   5.0|2000-07-07|
|B00002SSST|A1ETV7PW3286IM|   5.0|2000-07-08|
|B00000IWCT| A17N23WXA7EAF|   5.0|2000-07-12|
|B00001ZWV7| A8EDTKSPOMRWK|   5.0|2000-07-17|
|B00000IWCZ|A1WJI1ZNZ09D7B|   5.0|2000-07-20|
|B00004TFYY| AKQWL3SMUWL74|   4.0|2000-07-22|
|B00000IWFB|A18OEHKUGOENZZ|   5.0|

## Funções de Agrupamento

In [39]:
amz.show()

+----------+--------------+------+----------+
|      item|          user|rating|      date|
+----------+--------------+------+----------+
|0020232233|A1IDMI31WEANAF|   2.0|2016-09-22|
|0020232233| A4BCEVVZ4Y3V3|   1.0|2016-09-18|
|0020232233|A2EZ9PY1IHHBX0|   3.0|2016-09-12|
|0020232233|A139PXTTC2LGHZ|   5.0|2017-03-02|
|0020232233|A3IB33V29XIL8O|   1.0|2017-02-08|
|0020232233|A1J86V48S4KRJE|   5.0|2017-01-27|
|0020232233|A14J12PRBLGHF4|   5.0|2017-01-02|
|0020232233|A2UKOWP9ICU416|   5.0|2016-12-17|
|0020232233|A2ONKKDETRWT79|   4.0|2016-12-15|
|0020232233| AK9GN9KZZNTEP|   3.0|2016-12-09|
|0020232233|A26HO01PDWN6O5|   5.0|2016-12-07|
|0020232233| A8PF4X87CS3ZZ|   5.0|2016-11-05|
|0020232233|A28QGCUNS22JPH|   5.0|2016-11-04|
|038536539X|A2X2RJWNAV4FM0|   2.0|2017-04-24|
|038536539X|A1DIRJ8YNI5TKL|   2.0|2017-04-17|
|038536539X|A2G5IDD919P2DK|   5.0|2017-01-26|
|0486277577| AGCAAWP1AVVZR|   4.0|2004-03-16|
|0486277577| AD2CIUI4TSVY7|   5.0|2002-04-28|
|0486277577| APTEPFKL03QA0|   5.0|

In [40]:
# média de avaliações
avg_rating = amz.groupBy('item').avg('rating')
avg_rating.show(5)

+----------+------------------+
|      item|       avg(rating)|
+----------+------------------+
|1579823645| 4.610714285714286|
|1589949358|4.2555555555555555|
|B00000IUX6| 4.636363636363637|
|B00000IUI1| 4.181818181818182|
|B00000IZHN|               4.7|
+----------+------------------+
only showing top 5 rows


In [41]:
avg_rating.count()

7583

In [44]:
# ordenar do maior para menor valor

avg_rating.sort(desc('avg(rating)')).show(10)


+----------+-----------+
|      item|avg(rating)|
+----------+-----------+
|B0000E2DK4|        5.0|
|B00008MIIR|        5.0|
|B0006Q0DFK|        5.0|
|B00005CFAO|        5.0|
|B0002261TU|        5.0|
|B00003ABUL|        5.0|
|B00030EVDE|        5.0|
|B0006SK8CQ|        5.0|
|B000063KCQ|        5.0|
|B00000JQ4U|        5.0|
+----------+-----------+
only showing top 10 rows


In [46]:
cnt = amz.groupBy('item').count()

In [47]:
cnt.show()

+----------+-----+
|      item|count|
+----------+-----+
|1579823645|  280|
|1589949358|   90|
|B00000IUX6|   33|
|B00000IUI1|   11|
|B00000IZHN|   20|
|B0000296YQ|    2|
|B00003ABUL|   11|
|B00005BP3O|   26|
|B00009QMQR|  242|
|B0000WS00A|   26|
|B00012TL34|   56|
|B0001OM1JS|    6|
|B000219X64|   15|
|B0002TT3KM|   18|
|B00030EVDE|    2|
|B00065ARIY|    3|
|B0006OCFC6|   13|
|B0007LQGQ4|   27|
|B0007Y4DLG| 1135|
|1579822177|   79|
+----------+-----+
only showing top 20 rows


In [50]:
cnt_sorted = cnt.sort(desc('count'))

In [51]:
cnt_sorted.show()

+----------+-----+
|      item|count|
+----------+-----+
|0975277324| 4159|
|B000GUGY1S| 3904|
|0976990709| 3087|
|B000197NXM| 3042|
|B00000K3BR| 2944|
|B0000683A4| 2882|
|B000BN8Y8G| 2762|
|1933054395| 2711|
|B00068Q7LC| 2692|
|B00000J0S3| 2676|
|B00000ISC5| 2246|
|1932188126| 2228|
|B00005TQI7| 2178|
|B0006O8Q7Y| 2176|
|B00004TFT1| 2141|
|B00000IVAK| 2122|
|B00000IZJB| 2065|
|B00004YO0Y| 2029|
|157982319X| 2002|
|B000066665| 1913|
+----------+-----+
only showing top 20 rows


# Load: Escrita de Arquivos Parquet

In [52]:
amz.show()

+----------+--------------+------+----------+
|      item|          user|rating|      date|
+----------+--------------+------+----------+
|0020232233|A1IDMI31WEANAF|   2.0|2016-09-22|
|0020232233| A4BCEVVZ4Y3V3|   1.0|2016-09-18|
|0020232233|A2EZ9PY1IHHBX0|   3.0|2016-09-12|
|0020232233|A139PXTTC2LGHZ|   5.0|2017-03-02|
|0020232233|A3IB33V29XIL8O|   1.0|2017-02-08|
|0020232233|A1J86V48S4KRJE|   5.0|2017-01-27|
|0020232233|A14J12PRBLGHF4|   5.0|2017-01-02|
|0020232233|A2UKOWP9ICU416|   5.0|2016-12-17|
|0020232233|A2ONKKDETRWT79|   4.0|2016-12-15|
|0020232233| AK9GN9KZZNTEP|   3.0|2016-12-09|
|0020232233|A26HO01PDWN6O5|   5.0|2016-12-07|
|0020232233| A8PF4X87CS3ZZ|   5.0|2016-11-05|
|0020232233|A28QGCUNS22JPH|   5.0|2016-11-04|
|038536539X|A2X2RJWNAV4FM0|   2.0|2017-04-24|
|038536539X|A1DIRJ8YNI5TKL|   2.0|2017-04-17|
|038536539X|A2G5IDD919P2DK|   5.0|2017-01-26|
|0486277577| AGCAAWP1AVVZR|   4.0|2004-03-16|
|0486277577| AD2CIUI4TSVY7|   5.0|2002-04-28|
|0486277577| APTEPFKL03QA0|   5.0|

In [53]:
amz.write.parquet("amz_partitioned.parquet")

In [54]:
amz.coalesce(1).write.parquet("amz.parquet")

In [55]:
amz = spark.read.parquet('amz.parquet')

In [56]:
amz.printSchema()

root
 |-- item: string (nullable = true)
 |-- user: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- date: date (nullable = true)



In [57]:
amz.show(10)

+----------+--------------+------+----------+
|      item|          user|rating|      date|
+----------+--------------+------+----------+
|0020232233|A1IDMI31WEANAF|   2.0|2016-09-22|
|0020232233| A4BCEVVZ4Y3V3|   1.0|2016-09-18|
|0020232233|A2EZ9PY1IHHBX0|   3.0|2016-09-12|
|0020232233|A139PXTTC2LGHZ|   5.0|2017-03-02|
|0020232233|A3IB33V29XIL8O|   1.0|2017-02-08|
|0020232233|A1J86V48S4KRJE|   5.0|2017-01-27|
|0020232233|A14J12PRBLGHF4|   5.0|2017-01-02|
|0020232233|A2UKOWP9ICU416|   5.0|2016-12-17|
|0020232233|A2ONKKDETRWT79|   4.0|2016-12-15|
|0020232233| AK9GN9KZZNTEP|   3.0|2016-12-09|
+----------+--------------+------+----------+
only showing top 10 rows


In [58]:
amz.count()

500000

# Transformações Avançadas

## Join

### Leitura de Metadados (Parquet)

In [59]:
!ls '/content/drive/My Drive/amz/'

meta_small.json		     Toys_and_Games.csv
meta_Toys_and_Games.json     Toys_and_Games.parquet
meta_Toys_and_Games.parquet  Toys_and_Games_reviews_clean.json
small.csv		     Toys_and_Games_reviews.json
small.json


In [61]:
!tail '/content/drive/My Drive/amz/meta_Toys_and_Games.json'

{"category": ["Toys & Games", "Toy Remote Control & Play Vehicles", "Play Vehicles"], "description": ["Enjoy even larger scale Wheel Action Drivers excitement with this oversized vehicle inspired by Disney/Pixar Cars 3. Frank is a fan favorite and now you can recreate his tenacious hunt for trespassers with this bigger, badder Power Punch vehicle. It features his classic left and right thresher moves and a head that bobs forward with steely determination. Fans of all sizes can recreate the animated action of this and other favorites characters with our full assortment of wheel popping, big personality, large scale Wheel Action Drivers! Comes in \"Try My\" packaging to check the action out in advance. Each vehicle sold separately, subject to availability."], "title": "Disney/Pixar Cars Wheel Action Drivers Frank Vehicle", "image": ["https://images-na.ssl-images-amazon.com/images/I/51FV-wYc4oL._SS40_.jpg"], "brand": "Disney", "feature": ["Large scale Wheel Action Drivers vehicle inspired

In [62]:
meta = spark.read.parquet('/content/drive/My Drive/amz/meta_Toys_and_Games.parquet')

In [63]:
meta.printSchema()

root
 |-- prod: string (nullable = true)
 |-- code: string (nullable = true)
 |-- brand: string (nullable = true)



In [64]:
meta.show()

+--------------------+----------+--------------------+
|                prod|      code|               brand|
+--------------------+----------+--------------------+
|Dr. Suess 19163 D...|0000191639|           Dr. Seuss|
|Pathfinder: Book ...|0004950763|Pathfinder Rolepl...|
|Nursery Rhymes Fe...|0005069491|        Betty Lukens|
|Dutch Blitz Card ...|0004983289|Dutch Blitz Games Co|
|Magic Pen Paintin...|0006466222|    Lee Publications|
|Dungeons & Dragon...|0020232233|     Gale Force Nine|
|NUM NOMS figures ...|0096737581|            Num Noms|
|UDI U806 Infrared...|014002316X|                 UDI|
|Stellaluna Finger...|0152014764|         Design Farm|
|Oxford Ort Packs ...|019848710X|    Oxford Ort Packs|
|Oxford University...|0198487126|Oxford University...|
|Taito - Peluche D...|0298771357|               Taito|
|Nemuneko Big Plus...|0298772221|               Furyu|
|Zou no Pororon Bi...|0298752530|               Amuse|
|Touken Ranbu Onli...|0298752697|               Eikoh|
|Pokemon X

In [65]:
meta.count()

627539

### Leitura de Avaliações (Parquet)

In [66]:
amz = spark.read.parquet('/content/drive/My Drive/amz/Toys_and_Games.parquet')

In [67]:
amz.printSchema()

root
 |-- code: string (nullable = true)
 |-- user: string (nullable = true)
 |-- eval: double (nullable = true)



In [68]:
amz.show()

+----------+--------------+----+
|      code|          user|eval|
+----------+--------------+----+
|0020232233|A1IDMI31WEANAF| 2.0|
|0020232233| A4BCEVVZ4Y3V3| 1.0|
|0020232233|A2EZ9PY1IHHBX0| 3.0|
|0020232233|A139PXTTC2LGHZ| 5.0|
|0020232233|A3IB33V29XIL8O| 1.0|
|0020232233|A1J86V48S4KRJE| 5.0|
|0020232233|A14J12PRBLGHF4| 5.0|
|0020232233|A2UKOWP9ICU416| 5.0|
|0020232233|A2ONKKDETRWT79| 4.0|
|0020232233| AK9GN9KZZNTEP| 3.0|
|0020232233|A26HO01PDWN6O5| 5.0|
|0020232233| A8PF4X87CS3ZZ| 5.0|
|0020232233|A28QGCUNS22JPH| 5.0|
|038536539X|A2X2RJWNAV4FM0| 2.0|
|038536539X|A1DIRJ8YNI5TKL| 2.0|
|038536539X|A2G5IDD919P2DK| 5.0|
|0486277577| AGCAAWP1AVVZR| 4.0|
|0486277577| AD2CIUI4TSVY7| 5.0|
|0486277577| APTEPFKL03QA0| 5.0|
|0486277577|A14FXEKB1PXTZG| 5.0|
+----------+--------------+----+
only showing top 20 rows


In [69]:
amz.count()

8201231

In [75]:
amz.count()

8201231

### Operação de Join (União)

In [77]:
amz_with_metadata = amz.join(meta, on="code", how="inner")

In [78]:
amz_with_metadata.printSchema()

root
 |-- code: string (nullable = true)
 |-- user: string (nullable = true)
 |-- eval: double (nullable = true)
 |-- prod: string (nullable = true)
 |-- brand: string (nullable = true)



In [79]:
amz_with_metadata.show(50)

+----------+--------------+----+--------------------+--------------------+
|      code|          user|eval|                prod|               brand|
+----------+--------------+----+--------------------+--------------------+
|0000191639| AMEVO2LY6VEJA| 5.0|Dr. Suess 19163 D...|           Dr. Seuss|
|0004950763|A3H4ZZTMCQK1A2| 5.0|Pathfinder: Book ...|Pathfinder Rolepl...|
|0298770016|A3I052Z7Y5JQW2| 5.0|Nemuneko Honey He...|               Furyu|
|0298771861|A2VU4AWF9MGDP5| 5.0|Sega Love Live Sc...|                Sega|
|0298771861|A3OUPB3SW1NBKG| 5.0|Sega Love Live Sc...|                Sega|
|0298771861| ASX9QHZ393709| 5.0|Sega Love Live Sc...|                Sega|
|0298772221|A38Q3XNSFL81BT| 5.0|Nemuneko Big Plus...|               Furyu|
|0298772221|A2TIQGSXAZCQYI| 5.0|Nemuneko Big Plus...|               Furyu|
|0399232133|A1CXRAI6C7DD7B| 5.0|Goodnight, Gorill...|     Putnam Juvenile|
|0486402029|A1X9QQFMPGDW70| 1.0|     Pirates Tattoos|               Dover|
|0486402029|A1NTAPB1XPB6K

In [76]:
amz_with_metadata.count()

8427595

### Média por Produto

In [85]:
avg_prod_review = amz_with_metadata.groupBy('brand').avg('eval')

In [86]:
sorted = avg_prod_review.orderBy(desc('avg(eval)'))

In [87]:
sorted.show()

+--------------------+---------+
|               brand|avg(eval)|
+--------------------+---------+
|              Kutuwa|      5.0|
|       PRIMA FLOWERS|      5.0|
|             Mercury|      5.0|
|    United Chemi-Con|      5.0|
|V 320 - 1962 diec...|      5.0|
|VAN AKEN INTERNAT...|      5.0|
|Discount Dance Su...|      5.0|
|ZCWO Premier Coll...|      5.0|
|                Olli|      5.0|
|Tandy Corporation...|      5.0|
|  Douglas Wheel Tire|      5.0|
|      McDonald Corp.|      5.0|
| Valvigi Partnership|      5.0|
|Fiesta Toy Sea an...|      5.0|
|by\n    \n    Kin...|      5.0|
|Monterey Bay Aqua...|      5.0|
|  General Instrument|      5.0|
|Use Your Head Unl...|      5.0|
|                   -|      5.0|
|Disney Cars Jump ...|      5.0|
+--------------------+---------+
only showing top 20 rows


## Agregação com multiplos campos





In [88]:
# Agregar por marcar e calcular média de avaliações,
# contagem de produtos, e contagem de avaliações
agg_brand = amz_with_metadata.groupBy("brand").agg(
    avg("eval"),
    count_distinct("prod"),
    count("eval")
)

In [89]:
agg_brand.printSchema()

root
 |-- brand: string (nullable = true)
 |-- avg(eval): double (nullable = true)
 |-- count(DISTINCT prod): long (nullable = false)
 |-- count(eval): long (nullable = false)



In [90]:
agg_brand.show(5)

+--------------------+------------------+--------------------+-----------+
|               brand|         avg(eval)|count(DISTINCT prod)|count(eval)|
+--------------------+------------------+--------------------+-----------+
|            PlanToys| 4.317833471416736|                 470|       9656|
|H&F International...| 4.666666666666667|                   1|          3|
|           Happyfans|2.3333333333333335|                   2|          3|
|              XIDAJE| 2.963636363636364|                   7|         55|
|            Feldherr| 4.449056603773585|                  57|        530|
+--------------------+------------------+--------------------+-----------+
only showing top 5 rows


### Renomeação

In [91]:
# Renomear colunas para nomes mais amigaveis
agg_brand_clean = agg_brand \
    .withColumnRenamed("avg(eval)", "avg_eval") \
    .withColumnRenamed("count(DISTINCT prod)", "num_products") \
    .withColumnRenamed("count(eval)", "num_reviews")

In [92]:
agg_brand_clean.printSchema()

root
 |-- brand: string (nullable = true)
 |-- avg_eval: double (nullable = true)
 |-- num_products: long (nullable = false)
 |-- num_reviews: long (nullable = false)



In [93]:
agg_brand_clean.show()

+--------------------+------------------+------------+-----------+
|               brand|          avg_eval|num_products|num_reviews|
+--------------------+------------------+------------+-----------+
|            PlanToys| 4.317833471416736|         470|       9656|
|H&F International...| 4.666666666666667|           1|          3|
|           Happyfans|2.3333333333333335|           2|          3|
|              XIDAJE| 2.963636363636364|           7|         55|
|            Feldherr| 4.449056603773585|          57|        530|
|        A. Dougherty|               5.0|           1|          6|
|       Merry Toy Co.|               4.0|           1|          5|
|                 PMK|2.7857142857142856|           3|         14|
|     Powerpuff Girls| 4.205128205128205|          19|        117|
|            Rawlings| 4.178960096735188|          26|        827|
|         bestpriceam|3.4782608695652173|           8|         92|
|      Playmates Toys|  4.27536231884058|          20|        

### Renomeação com select

In [95]:
# As colunas podem ser renomeadas na seleção
agg_brand_clean = agg_brand.select(
    col("brand"),
    col("avg(eval)").alias("avg_eval"),
    col("count(DISTINCT prod)").alias("num_products"),
    col("count(eval)").alias("num_reviews")
)

In [96]:
agg_brand_clean.show()

+--------------------+------------------+------------+-----------+
|               brand|          avg_eval|num_products|num_reviews|
+--------------------+------------------+------------+-----------+
|            PlanToys| 4.317833471416736|         470|       9656|
|H&F International...| 4.666666666666667|           1|          3|
|           Happyfans|2.3333333333333335|           2|          3|
|              XIDAJE| 2.963636363636364|           7|         55|
|            Feldherr| 4.449056603773585|          57|        530|
|        A. Dougherty|               5.0|           1|          6|
|       Merry Toy Co.|               4.0|           1|          5|
|                 PMK|2.7857142857142856|           3|         14|
|     Powerpuff Girls| 4.205128205128205|          19|        117|
|            Rawlings| 4.178960096735188|          26|        827|
|         bestpriceam|3.4782608695652173|           8|         92|
|      Playmates Toys|  4.27536231884058|          20|        

### Arredondamento

In [97]:
# Arredondamento da média com duas casas decimais
agg_brand_clean = agg_brand_clean.withColumn("avg_eval", round(col("avg_eval"), 2))

In [98]:
agg_brand_clean.show()

+--------------------+--------+------------+-----------+
|               brand|avg_eval|num_products|num_reviews|
+--------------------+--------+------------+-----------+
|            PlanToys|    4.32|         470|       9656|
|H&F International...|    4.67|           1|          3|
|           Happyfans|    2.33|           2|          3|
|              XIDAJE|    2.96|           7|         55|
|            Feldherr|    4.45|          57|        530|
|        A. Dougherty|     5.0|           1|          6|
|       Merry Toy Co.|     4.0|           1|          5|
|                 PMK|    2.79|           3|         14|
|     Powerpuff Girls|    4.21|          19|        117|
|            Rawlings|    4.18|          26|        827|
|         bestpriceam|    3.48|           8|         92|
|      Playmates Toys|    4.28|          20|         69|
|         Yummy World|    4.77|           7|         13|
|         ShalinIndia|    3.73|          96|       1277|
|               FUTAO|    4.29|

### Filtros

In [99]:
agg_brand_clean.count()

62565

In [100]:
# Filtrar marcas com mais de 10 produtos e mais de 100 avaliações
filtered = agg_brand_clean.filter(
    (agg_brand_clean.num_products > 10)
    & (agg_brand_clean.num_reviews > 100))
filtered.show()


+--------------------+--------+------------+-----------+
|               brand|avg_eval|num_products|num_reviews|
+--------------------+--------+------------+-----------+
|            PlanToys|    4.32|         470|       9656|
|            Feldherr|    4.45|          57|        530|
|     Powerpuff Girls|    4.21|          19|        117|
|            Rawlings|    4.18|          26|        827|
|         ShalinIndia|    3.73|          96|       1277|
|Stuffed Animal House|    4.25|          14|        138|
|            Warcraft|    4.24|          88|        594|
|         Wild Planet|    3.58|         116|       2500|
|       Swing-N-Slide|    4.21|          95|       4648|
|               AVAWO|     4.3|          22|       1269|
|      Fun To Collect|     3.8|          41|        130|
|              gloria|    3.64|          35|       1404|
|        NW education|    3.97|          23|        110|
|            Hoberman|    4.09|          17|        656|
|     Bloco Toys inc.|    4.15|

In [101]:
filtered.count()

3488

### Ordenação

In [102]:
# Ordenado por numero de produtos
filtered.orderBy(desc("num_products")).show(truncate=False)

+--------------------+--------+------------+-----------+
|brand               |avg_eval|num_products|num_reviews|
+--------------------+--------+------------+-----------+
|Disney              |4.16    |10968       |98474      |
|Mattel              |4.25    |8083        |119436     |
|Yu-Gi-Oh!           |4.57    |8029        |38838      |
|Magic: The Gathering|4.49    |7864        |32313      |
|LEGO                |4.63    |7050        |211895     |
|Pokemon             |4.29    |6297        |49731      |
|Hasbro              |4.21    |6101        |144751     |
|Amscan              |4.1     |5216        |41031      |
|Fisher-Price        |4.35    |4488        |213142     |
|Bandai              |4.44    |4259        |30175      |
|Hot Wheels          |4.23    |3956        |40030      |
|Barbie              |4.4     |3760        |72042      |
|FunKo               |4.48    |3751        |120312     |
|Fun Express         |3.83    |3501        |88337      |
|Star Wars           |4.19    |

In [103]:
# Ordenado por numero de avaliações
filtered.orderBy(desc("num_reviews")).show(truncate=False)

+--------------------+--------+------------+-----------+
|brand               |avg_eval|num_products|num_reviews|
+--------------------+--------+------------+-----------+
|Melissa & Doug      |4.44    |2429        |217687     |
|Fisher-Price        |4.35    |4488        |213142     |
|LEGO                |4.63    |7050        |211895     |
|Hasbro              |4.21    |6101        |144751     |
|FunKo               |4.48    |3751        |120312     |
|Mattel              |4.25    |8083        |119436     |
|Disney              |4.16    |10968       |98474      |
|VTech               |4.29    |893         |95767      |
|Fun Express         |3.83    |3501        |88337      |
|Crayola             |4.13    |1521        |77225      |
|Barbie              |4.4     |3760        |72042      |
|Little Tikes        |4.25    |731         |63468      |
|LeapFrog            |4.31    |682         |62374      |
|Nerf                |4.17    |506         |57262      |
|Rhode Island Novelty|3.6     |

In [106]:
# Ordenado por avaliação média (descendente)
filtered.filter(col('num_products') > 500).orderBy(col("avg_eval")).show(truncate=False)

+--------------------+--------+------------+-----------+
|brand               |avg_eval|num_products|num_reviews|
+--------------------+--------+------------+-----------+
|Velocity Toys       |3.15    |1073        |10081      |
|WeGlow International|3.53    |637         |1453       |
|Rhode Island Novelty|3.6     |1638        |57227      |
|Century Novelty     |3.62    |557         |3446       |
|U.S. Toy            |3.65    |976         |14483      |
|Forum Novelties     |3.72    |850         |13597      |
|Little Treasures    |3.73    |846         |4773       |
|Redcat Racing       |3.76    |592         |6965       |
|Generic             |3.77    |1785        |15220      |
|Integy              |3.8     |892         |1688       |
|Toys R Us           |3.81    |627         |2454       |
|Toysmith            |3.81    |1207        |36953      |
|Fun Express         |3.83    |3501        |88337      |
|Spin Master         |3.85    |1138        |12998      |
|Unknown             |3.86    |

In [105]:
# Ordenado por avaliação média
filtered.orderBy("avg_eval").show(truncate=False)

+----------------------+--------+------------+-----------+
|brand                 |avg_eval|num_products|num_reviews|
+----------------------+--------+------------+-----------+
|AMG                   |1.99    |18          |205        |
|3DIT Character Creator|2.12    |13          |196        |
|Toy Guitars           |2.21    |20          |225        |
|Girl Gourmet          |2.25    |11          |197        |
|Toys4less             |2.27    |62          |166        |
|Merchsource           |2.3     |33          |255        |
|HIHAOXJ               |2.3     |31          |129        |
|Kole                  |2.31    |43          |104        |
|Saitek                |2.33    |11          |217        |
|NSI                   |2.36    |57          |959        |
|Good Old Values       |2.38    |25          |295        |
|Toy Quest             |2.43    |24          |148        |
|Arbor Toys Co. LTD    |2.49    |13          |101        |
|Incredible Science    |2.52    |13          |312       

## Criação de índices

### Leitura de arquivo JSON com esquema definido

In [107]:
!tail '/content/drive/My Drive/amz/Toys_and_Games_reviews.json'

{"image": ["https://images-na.ssl-images-amazon.com/images/I/818PSrnxCeL._SY88.jpg", "https://images-na.ssl-images-amazon.com/images/I/71Z+BdiPyvL._SY88.jpg"], "overall": 4.0, "verified": true, "reviewTime": "04 25, 2018", "reviewerID": "A1INIIHKD63JXZ", "asin": "B01HJBAKIO", "reviewerName": "B421", "reviewText": "I wish this were a die cast but my son loves it the way it is.  Frank is a hard to find Cars vehicle.  Very happy with this purchase.", "summary": "Frank cake 3rd Birthday.", "unixReviewTime": 1524614400}
{"overall": 5.0, "verified": true, "reviewTime": "03 30, 2018", "reviewerID": "A1W75PEIIL0IFM", "asin": "B01HJBAKIO", "reviewerName": "Perlaaaa ", "reviewText": "My son LOVES IT!! Frank is a lot bigger than I expected which is awesome! Shipped right away!", "summary": "Great Purchase!", "unixReviewTime": 1522368000}
{"overall": 4.0, "verified": true, "reviewTime": "03 14, 2018", "reviewerID": "A3I38QQ7ARQD8F", "asin": "B01HJBAKIO", "reviewerName": "Shannon Pedigo", "reviewTe

In [108]:
# Define the schema with only the desired columns
schema = StructType([
    StructField("asin", StringType(), True),
    StructField("reviewText", StringType(), True),
])


In [ ]:
# text_reviews = spark.read \
#     .option("multiLine", False) \
#     .option("mode", "DROPMALFORMED") \
#     .schema(schema) \
#     .json("/content/drive/My Drive/amz/Toys_and_Games_reviews.json")



In [109]:
text_reviews = spark.read \
    .option("multiLine", False) \
    .option("mode", "DROPMALFORMED") \
    .schema(schema) \
    .json("/content/drive/My Drive/amz/small.json")

In [110]:
text_reviews.show(5, truncate=50)


+----------+--------------------------------------------------+
|      asin|                                        reviewText|
+----------+--------------------------------------------------+
|0020232233|When it comes to a DM's screen, the space on th...|
|0020232233|An Open Letter to GaleForce9*:\n\nYour unpainte...|
|0020232233|Nice art, nice printing.  Why two panels are fi...|
|0020232233|Amazing buy! Bought it as a gift for our new dm...|
|0020232233|As my review of GF9's previous screens these we...|
+----------+--------------------------------------------------+
only showing top 5 rows


### Limpeza do texto

In [111]:
text_reviews = text_reviews.withColumn("reviewText", lower(col("reviewText")))
text_reviews.show()

+----------+--------------------+
|      asin|          reviewText|
+----------+--------------------+
|0020232233|when it comes to ...|
|0020232233|an open letter to...|
|0020232233|nice art, nice pr...|
|0020232233|amazing buy! boug...|
|0020232233|as my review of g...|
|0020232233|      grandson loves|
|0020232233|i have bought man...|
|0020232233|came in perfect c...|
|0020232233|could be better b...|
|0020232233|my review will mi...|
|0020232233|     works very well|
|0020232233|can't wait to use...|
|0020232233|this is a campaig...|
|038536539X|this is one of th...|
|038536539X|it sounded like a...|
|038536539X|very fun game for...|
|0486277577|pretty good book ...|
|0486277577|when i unexpected...|
|0486277577|if you've mastere...|
|0486277577|i've yet to see a...|
+----------+--------------------+
only showing top 20 rows


In [112]:
cleaned = text_reviews.withColumn(
    "cleanText",
    regexp_replace("reviewText", "[^a-z ]+", "")
)

In [113]:
cleaned.show(truncate=50)

+----------+--------------------------------------------------+--------------------------------------------------+
|      asin|                                        reviewText|                                         cleanText|
+----------+--------------------------------------------------+--------------------------------------------------+
|0020232233|when it comes to a dm's screen, the space on th...|when it comes to a dms screen the space on the ...|
|0020232233|an open letter to galeforce9*:\n\nyour unpainte...|an open letter to galeforceyour unpainted minia...|
|0020232233|nice art, nice printing.  why two panels are fi...|nice art nice printing  why two panels are fill...|
|0020232233|amazing buy! bought it as a gift for our new dm...|amazing buy bought it as a gift for our new dm ...|
|0020232233|as my review of gf9's previous screens these we...|as my review of gfs previous screens these were...|
|0020232233|                                    grandson loves|                 

### Separação de palavras

In [114]:
tokenized = cleaned.withColumn("words", split(col("cleanText"), " "))

In [115]:
tokenized.show(truncate=50)

+----------+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+
|      asin|                                        reviewText|                                         cleanText|                                             words|
+----------+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+
|0020232233|when it comes to a dm's screen, the space on th...|when it comes to a dms screen the space on the ...|[when, it, comes, to, a, dms, screen, the, spac...|
|0020232233|an open letter to galeforce9*:\n\nyour unpainte...|an open letter to galeforceyour unpainted minia...|[an, open, letter, to, galeforceyour, unpainted...|
|0020232233|nice art, nice printing.  why two panels are fi...|nice art nice printing  why two panels are fill...|[nice, art, nice, printing, , why, two, panels,...|
|002

In [116]:
tokenized.printSchema()

root
 |-- asin: string (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- cleanText: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = false)



In [117]:
tokenized.count()

100000

In [118]:
exploded = tokenized.select("asin", explode("words").alias("word"))

In [119]:
exploded.show()

+----------+--------+
|      asin|    word|
+----------+--------+
|0020232233|    when|
|0020232233|      it|
|0020232233|   comes|
|0020232233|      to|
|0020232233|       a|
|0020232233|     dms|
|0020232233|  screen|
|0020232233|     the|
|0020232233|   space|
|0020232233|      on|
|0020232233|     the|
|0020232233|  screen|
|0020232233|  itself|
|0020232233|      is|
|0020232233|      at|
|0020232233|      an|
|0020232233|absolute|
|0020232233| premium|
|0020232233|     the|
|0020232233|    fact|
+----------+--------+
only showing top 20 rows


In [120]:
exploded.count()

4725910

### Filtragem de palavras curtas

In [121]:
filtered = exploded.filter(length("word") > 3)


In [122]:
filtered.show()

+----------+-----------+
|      asin|       word|
+----------+-----------+
|0020232233|       when|
|0020232233|      comes|
|0020232233|     screen|
|0020232233|      space|
|0020232233|     screen|
|0020232233|     itself|
|0020232233|   absolute|
|0020232233|    premium|
|0020232233|       fact|
|0020232233|       that|
|0020232233|       this|
|0020232233|      space|
|0020232233|     wasted|
|0020232233|   terribly|
|0020232233|informative|
|0020232233|     needed|
|0020232233|       well|
|0020232233|      makes|
|0020232233| completely|
|0020232233|    useless|
+----------+-----------+
only showing top 20 rows


In [123]:
filtered.count()

2527846

### Busca por termos

In [124]:
filtered.filter(col("word") == "battery").show()

+----------+-------+
|      asin|   word|
+----------+-------+
|0994606710|battery|
|0994606710|battery|
|0994606737|battery|
|1223063151|battery|
|1223063151|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
|157982319X|battery|
+----------+-------+
only showing top 20 rows


In [125]:
# Produtos cujas avaliações mencionam "battery"
filtered.filter(col("word") == "battery").select("asin").distinct().show()


+----------+
|      asin|
+----------+
|B00000IS02|
|9986255198|
|157982319X|
|7293000014|
|9269802566|
|8677809945|
|B00000ISR7|
|0994606710|
|8663509472|
|9269802590|
|849900220X|
|B00000IU72|
|B00000IUD2|
|B00000IWIP|
|9269808416|
|B00000IVZJ|
|B00000ISKD|
|B00000IUCE|
|B00000ISEH|
|9989956383|
+----------+
only showing top 20 rows


In [126]:
filtered.filter(col("word") == "battery").select("asin").distinct().count()


88

In [ ]:
# Produtos cujas avaliações mencionam "game"
filtered.filter(col("word") == "game").select("asin").distinct().show()
filtered.filter(col("word") == "game").select("asin").distinct().count()


### Termos mais comuns

In [ ]:
filtered.groupBy("word").count().show()

In [ ]:
top_words = filtered.groupBy("word").count().orderBy(col("count").desc()).limit(10)
top_words.show(truncate=False)

## Exemplo Avançado: TF-IDF

### Termos por documento

In [ ]:
doc_total_words = filtered.groupBy("asin").count().withColumnRenamed("count", "total_words")


In [ ]:
doc_total_words.show()

In [ ]:
num_docs = doc_total_words.count()

### Frequência dos Termos (TF)

In [ ]:
# Agrupamento (chave) por asin+palavra
tf = filtered.groupBy("asin", "word").count().withColumnRenamed("count", "tf")


In [ ]:
tf.orderBy(desc("tf")).show()

### Normalização

In [ ]:
tf_with_total = tf.join(doc_total_words, on="asin")


In [ ]:
tf_with_total.show()

In [ ]:
tf_normalized = tf_with_total.withColumn(
    "tf_norm",
    col("tf") / col("total_words")
)

In [ ]:
tf_normalized.orderBy(desc('tf_norm')).show()

### Frequência nos Documentos (DF)

In [ ]:
df = filtered.groupBy("word").agg(
    count_distinct("asin").alias("df")
)

In [ ]:
df.show()

### Inverso da frequência nos documentos (IDF)

In [ ]:
idf = df.withColumn("idf",
    log(num_docs / col("df"))
)

In [ ]:
idf.show()

In [ ]:
# Filtro para eliminar palavras presentes em poucos documentos
idf = idf.filter(col("df") >= 5)
idf.show()

### TF-IDF

In [ ]:
tfidf = tf_normalized.join(idf, on="word")


In [ ]:
tfidf.show()

In [ ]:
tfidf = tfidf.withColumn("tfidf", col("tf_norm") * col("idf"))

In [ ]:
tfidf.orderBy(desc("tfidf")).show()

### Termo mais descritivo por documento

In [ ]:

max_tfidf = tfidf.groupBy("asin").max("tfidf")


In [ ]:
max_tfidf.show()

In [ ]:
top_words = tfidf.join(
    max_tfidf,
    (tfidf["asin"] == max_tfidf["asin"]) & (tfidf["tfidf"] == max_tfidf["max(tfidf)"])
)

In [ ]:
top_words.show()